# Music-to-Dance Generation via Atomic Movements — Google Colab

Runs the full two-stage pipeline (atomic planner → dance completion) on a Colab
GPU: setup, dataset download, training, inference, rendering, and evaluation.

**Before running:** pick a GPU runtime via *Runtime → Change runtime type →
Hardware accelerator → GPU* (a T4 is enough).

Colab VMs are wiped when the session ends. The *Keep results in Google Drive*
section below mirrors checkpoints and outputs to your Drive so a disconnect does
not cost you a training run.

## 1. Set up the environment

The clone below decides which version of the code you run, independently of
where this notebook was opened from. If you opened the notebook from a branch,
set `BRANCH` to that same branch so the two match.

In [ ]:
!nvidia-smi || echo "No GPU detected - use Runtime > Change runtime type > GPU"

In [ ]:
import os

REPO_URL = "https://github.com/yamak493/AtomicDance.git"
REPO_DIR = "/content/AtomicDance"
# Set BRANCH to a feature branch to try changes that are not on main yet.
BRANCH = "main"

if not os.path.isdir(REPO_DIR):
    !git clone --branch "{BRANCH}" "{REPO_URL}" $REPO_DIR
%cd $REPO_DIR
!git log --oneline -1

Colab already provides a CUDA build of PyTorch, so `requirements-colab.txt`
installs only what is missing around it. The repository's `compat/` package
replaces PyTorch3D (no Colab wheels, tens of minutes to build) with pure
PyTorch rotation conversions, so there is nothing to compile here.

In [ ]:
!pip install -q -r requirements-colab.txt

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

Run the test suite to confirm the checkout works on this runtime before
spending GPU time on it.

In [ ]:
!python -m unittest discover -s tests -t . -v

## 2. Keep results in Google Drive (recommended)

Mount your Drive and keep `runs/` (checkpoints) and `outputs/` there, so a
disconnected session does not lose them. Skip this cell to keep everything on
the VM's local disk instead.

In [ ]:
from pathlib import Path

USE_DRIVE = True  # set to False to keep everything on the local VM disk

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/AtomicDance")
else:
    WORK_DIR = Path("/content/atomicdance-work")

RUNS_DIR = WORK_DIR / "runs"
OUTPUT_DIR = WORK_DIR / "outputs"
for directory in (RUNS_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("checkpoints ->", RUNS_DIR)
print("outputs     ->", OUTPUT_DIR)

## 3. Download the atomic dataset

The processed `atomic_aistpp` package holds the frame-aligned motion,
35-dimensional music features, and atomic labels. Labels `1..100` are movement
categories and `0` marks a transition.

If `gdown` hits Google Drive's quota, download the archive manually from the
[dataset link](https://drive.google.com/file/d/1ETsaetMMWeKV3_E3Lr40BdybAsUAG8WM/view?usp=sharing)
and extract it to `data/atomic_aistpp/`.

In [ ]:
DATASET_URL = "https://drive.google.com/file/d/1ETsaetMMWeKV3_E3Lr40BdybAsUAG8WM/view"

!pip install -q --upgrade gdown
!gdown --fuzzy "{DATASET_URL}" -O /content/atomic_aistpp.zip
!mkdir -p data
!unzip -q -o /content/atomic_aistpp.zip -d data/
!ls data/atomic_aistpp

In [ ]:
import json
from pathlib import Path

import numpy as np

root = Path("data/atomic_aistpp")
for split in ("train", "test"):
    motion = np.load(root / split / "motion.npy", mmap_mode="r")
    music = np.load(root / split / "music.npy", mmap_mode="r")
    labels = np.load(root / split / "labels.npy", mmap_mode="r")
    names = json.loads((root / split / "names.json").read_text())
    print(split, "motion", motion.shape, "music", music.shape,
          "labels", labels.shape, "names", len(names))

## 4. Train

Both stages read `--data-root data/atomic_aistpp`. Checkpoints are resumable:
pass `--resume <checkpoint>` to continue after a disconnect, which is the reason
for writing them to Drive.

Colab VMs have two CPU cores, so `--workers 2` loads faster than the default 4.
Batch sizes below fit a 16 GB T4; drop them if you hit out-of-memory.

Start with the smoke test to check the whole loop runs end to end in seconds.

In [ ]:
!python train_atomic.py \
  --stage planner \
  --data-root data/atomic_aistpp \
  --output-dir /tmp/smoke_planner \
  --device cuda \
  --epochs 1 --max-steps 2 --batch-size 2 --workers 2

### Atomic movement planner

In [ ]:
!python train_atomic.py \
  --stage planner \
  --data-root data/atomic_aistpp \
  --output-dir "{RUNS_DIR}/atomic_planner" \
  --device cuda \
  --epochs 20 \
  --batch-size 16 \
  --workers 2

### Dance completion model

200 epochs takes many hours and will outlast a free Colab session. Train in
stages: run this cell, then re-run it with `--resume` pointed at the newest
checkpoint under `RUNS_DIR/atomic_completion` (saved every 20 epochs).

In [ ]:
!python train_atomic.py \
  --stage completion \
  --data-root data/atomic_aistpp \
  --output-dir "{RUNS_DIR}/atomic_completion" \
  --device cuda \
  --epochs 200 \
  --batch-size 8 \
  --workers 2

In [ ]:
from pathlib import Path


def newest_checkpoint(directory):
    checkpoints = sorted(Path(directory).glob("*.pt"), key=lambda path: path.stat().st_mtime)
    if not checkpoints:
        raise FileNotFoundError("no checkpoint under {}".format(directory))
    return checkpoints[-1]


PLANNER_CHECKPOINT = newest_checkpoint(RUNS_DIR / "atomic_planner")
COMPLETION_CHECKPOINT = newest_checkpoint(RUNS_DIR / "atomic_completion")
print("planner   :", PLANNER_CHECKPOINT)
print("completion:", COMPLETION_CHECKPOINT)

## 5. Generate dances

Point `--audio-dir` at a folder of WAVs. The atomic dataset does not ship audio,
so either place the AIST++ WAVs under `data/edge_aistpp/wavs` (see section 7) or
upload your own with the cell below.

AIST++ filenames (`gWA_sBM_c01_d25_mWA4_ch05.wav`) let the feature extractor read
the tempo straight from the name; any other filename falls back to beat tracking,
which works just as well.

In [ ]:
AUDIO_DIR = "data/edge_aistpp/wavs"  # or str(CUSTOM_AUDIO_DIR) for your own music
GENERATED_DIR = OUTPUT_DIR / "generated"

!python infer_atomic.py \
  --audio-dir "{AUDIO_DIR}" \
  --output-dir "{GENERATED_DIR}" \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --data-root data/atomic_aistpp \
  --device cuda \
  --max-frames 150 \
  --inference-batch-size 4

In [ ]:
AUDIO_DIR = "data/edge_aistpp/wavs"  # or your own folder of .wav files
GENERATED_DIR = OUTPUT_DIR / "generated"

!python infer_atomic.py \
  --audio-dir "{AUDIO_DIR}" \
  --output-dir "{GENERATED_DIR}" \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --data-root data/atomic_aistpp \
  --device cuda \
  --max-frames 150 \
  --inference-batch-size 4

## 6. Render a result

`skeleton_render` writes a GIF of the skeleton and muxes it with the audio into
an MP4 (Colab already has ffmpeg). Rendering is CPU-bound, so keep the clip
short while you are iterating.

In [ ]:
import pickle
from pathlib import Path

import numpy as np

from vis import skeleton_render

generated = sorted(Path(GENERATED_DIR).glob("*.pkl"))
print("generated motions:", len(generated))

motion_path = generated[0]
with open(motion_path, "rb") as handle:
    data = pickle.load(handle)

frames = 150  # 5 seconds at 30 FPS
render_dir = OUTPUT_DIR / "renders"
skeleton_render(
    np.asarray(data["full_pose"])[:frames],
    epoch="atomic",
    out=str(render_dir),
    name=data["audio_path"],
    sound=True,
    contact=np.asarray(data["contacts"])[:frames],
)
print(sorted(path.name for path in render_dir.glob("*.mp4")))

In [ ]:
import base64
from pathlib import Path

from IPython.display import HTML

video_path = sorted(Path(render_dir).glob("*.mp4"))[0]
encoded = base64.b64encode(video_path.read_bytes()).decode()
HTML('<video width=480 controls><source src="data:video/mp4;base64,{}" type="video/mp4"></video>'.format(encoded))

## 7. Evaluate (optional)

Metrics compare generated motion against AIST++ ground truth, so this section
needs two assets that are not part of the project release and carry their own
licenses:

- AIST++ motion PKLs and WAVs under `data/edge_aistpp/{motions,wavs}`, from the
  [AIST++ website](https://google.github.io/aistplusplus_dataset/).
- `SMPL_MALE.pkl` at `smpl/SMPL_MALE.pkl`, from the
  [SMPL website](https://smpl.is.tue.mpg.de/).

The SMPL release stores its arrays as chumpy objects, and chumpy does not import
on Colab's Python. `compat/smpl.py` reads the file without it, so the licensed
`.pkl` works as downloaded.

The evaluator reports kinematic and manual-feature FID and diversity plus Beat
Alignment Score. Add `--overwrite-inference --force-extract` to rebuild the
generated motions and cached features instead of reusing them.

In [ ]:
!python -m eval.evaluate \
  --ground-truth-motions data/edge_aistpp/motions \
  --audio-dir data/edge_aistpp/wavs \
  --sequence-list data/splits/crossmodal_test.txt \
  --plan-source planner \
  --planner-checkpoint "{PLANNER_CHECKPOINT}" \
  --completion-checkpoint "{COMPLETION_CHECKPOINT}" \
  --atomic-data-root data/atomic_aistpp \
  --smpl-model smpl/SMPL_MALE.pkl \
  --device cuda:0 \
  --max-inference-frames 150 \
  --inference-batch-size 4 \
  --workers 2 \
  --inference-output "{OUTPUT_DIR}/eval_generated" \
  --cache-dir "{OUTPUT_DIR}/eval_cache" \
  --output "{OUTPUT_DIR}/results_planner.json"

In [ ]:
import json

print(json.dumps(json.loads((OUTPUT_DIR / "results_planner.json").read_text()), indent=2))

## Troubleshooting

**`CUDA out of memory`** — lower `--batch-size` (training) or
`--inference-batch-size`, then restart the runtime to release the allocator's
cached blocks.

**Session disconnected mid-training** — re-run the setup cells and pass
`--resume <checkpoint>` to `train_atomic.py`. This only works if the checkpoints
went to Drive.

**`gdown` quota exceeded** — download the dataset archive by hand from the link
in section 3 and extract it to `data/atomic_aistpp/`, or copy it into your Drive
and unzip it from there.

**`ModuleNotFoundError: No module named 'chumpy'`** — you are on a code path
that still calls `smplx` directly. Load the model through
`compat.smpl.load_smpl` instead.

**Rendering produces no MP4** — `skeleton_render` shells out to ffmpeg and
ignores its exit status. Check that the audio path in the PKL still exists on
this VM; re-run inference if the WAVs were downloaded to a wiped local disk.